<a href="https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/12_big_data_beyond_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- nav-header -->
[⬅ Previous](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/11_reinforcement_learning_qlearning.ipynb) · [🗺️ Course index](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb) · **Notebook 12 of the course** · [Next: 13 — Fairness & subgroup performance ➡](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/13_fairness_and_subgroups.ipynb)


# 🩺 Notebook 12 — When your data is too big for pandas *(bonus, reference)*

> Our teaching file is tiny (~10 MB). Real projects can be **gigabytes to terabytes**. This short
> reference explains **when pandas stops working** and **what to reach for instead** — plus where the
> really big datasets come from. Skim it, keep it as a cheat-sheet — and at the end, **run the same
> cleaning pipeline in four engines and time them**.

**In one line:** pandas loads everything into **RAM**, so it breaks when a dataset (× a few) no longer
fits in memory. The fixes are: *load less*, *use a better file format*, *process in chunks*, or *switch to
an out-of-core / parallel tool*.

**⏱️ Time:** ≈15 min to read · +10 min for the four-engine benchmark · **Level:** reference / take‑home

### ⚙️ Run me first

In [1]:
# === ⚙️  Workshop setup — run this cell first ===============================
# Works in Google Colab and in local Jupyter. Installs anything missing, sets a
# clean plotting style, and gives you helpers to load the data.
# (This cell is identical in every notebook of the course.)
import importlib.util, subprocess, sys, os, random, warnings
warnings.filterwarnings("ignore")

# --- reproducibility: everyone in the room gets the same numbers ---------------
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

# 📥 Where the workshop data comes from — already set up for you, nothing to do.
# The data downloads automatically the first time you need it. If you were given a
# different link, just paste it in place of the one below. These forms all work:
#   • a Google-Drive folder link      • a Drive / Dropbox / OneDrive file link
#   • a folder URL ending in "/"      • a link straight to a .zip
# Set it to "" if you would rather upload the CSVs by hand.
# (The data is not in the GitHub repo: it is real de-identified patient data covered
#  by a data use agreement and may not be redistributed openly.)
WORKSHOP_DATA_URL = os.environ.get(
    "WORKSHOP_DATA_URL",
    "https://drive.google.com/drive/folders/1y7CparhrqdlCAZq6xQlj8fZda394fniD")

def _ensure(pkgs):
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing])
_ensure({"numpy":"numpy","pandas":"pandas","sklearn":"scikit-learn",
         "matplotlib":"matplotlib","seaborn":"seaborn","shap":"shap","xgboost":"xgboost"})

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(RANDOM_STATE)          # seeds the legacy global np.random.* calls
RNG = np.random.default_rng(RANDOM_STATE)   # the modern generator — use this one
pd.set_option("display.max_columns", 120); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5); plt.rcParams["figure.dpi"] = 110

# Every model, split and resample in this course passes random_state=RANDOM_STATE, so
# your numbers should match your neighbour's exactly. (Different library *versions* can
# still shift the last decimal — that is normal and not a mistake on your part.)

# --- data loading: works locally AND remembers your upload across notebooks -----
_CACHE = {"dir": "unset"}   # memo so we only touch Google Drive once per session

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _drive_cache():
    """In Google Colab, mount Drive ONCE and return a persistent folder. A file you
    upload in one notebook is saved here, so every other notebook opens it automatically
    — no re-uploading. Returns None outside Colab, or if you decline to connect Drive."""
    if _CACHE["dir"] != "unset":
        return _CACHE["dir"]
    result = None
    if _in_colab():
        try:
            from google.colab import drive
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
            result = "/content/drive/MyDrive/sepsis_workshop_data"
            os.makedirs(result, exist_ok=True)
        except Exception:
            result = None
    _CACHE["dir"] = result
    return result

def _find(name):
    """Look for the file on disk. Deliberately does NOT touch Google Drive, so the normal
    path never triggers an authorisation popup."""
    for p in [name, f"data/{name}", f"../data/{name}", f"workshop/data/{name}"]:
        if os.path.exists(p):
            return p
    return None

def _find_in_drive(name):
    """Only used as a fallback, because it mounts Drive (and that means a popup)."""
    cache = _drive_cache()
    if cache:
        p = os.path.join(cache, name)
        if os.path.exists(p):
            return p
    return None

def _direct_url(u):
    """Turn an ordinary Google-Drive / Dropbox / OneDrive *share* link into one that a
    plain HTTP client can actually download, so you can paste the link you were given."""
    import re
    m = (re.search(r"drive\.google\.com/file/d/([\w-]+)", u)
         or re.search(r"drive\.google\.com/(?:open|uc)\?(?:export=\w+&)?id=([\w-]+)", u))
    if m:
        return f"https://drive.google.com/uc?export=download&id={m.group(1)}"
    if "dropbox.com" in u:
        return u.split("?")[0] + "?dl=1"
    if "sharepoint.com" in u or "1drv.ms" in u:
        return u + ("&" if "?" in u else "?") + "download=1"
    return u

def _fetch(url, dest):
    import urllib.request, shutil as _sh
    req = urllib.request.Request(_direct_url(url), headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
        _sh.copyfileobj(r, f)

_FOLDER = {"done": False}

def _gdrive_folder(name):
    """WORKSHOP_DATA_URL points at a Google-Drive *folder*: fetch it once with gdown
    (a folder cannot be downloaded with a plain HTTP request), then serve files from it."""
    dest = "_workshop_data"
    if not _FOLDER["done"]:
        _ensure({"gdown": "gdown"})
        import gdown
        print("⬇  fetching the workshop data from Google Drive (just once) …")
        gdown.download_folder(url=WORKSHOP_DATA_URL, output=dest, quiet=True, use_cookies=False)
        _FOLDER["done"] = True
    for root, _dirs, files in os.walk(dest):
        if name in files:
            return os.path.join(root, name)
    return None

def _try_download(name):
    """Fetch the data from WORKSHOP_DATA_URL, if one was configured."""
    u = (WORKSHOP_DATA_URL or "").strip()
    if not u:
        return None
    try:
        if "/drive/folders/" in u:
            return _gdrive_folder(name)
        if u.lower().split("?")[0].endswith(".zip"):
            import zipfile
            bundle = "_workshop_data.zip"
            if not os.path.exists(bundle):
                print("⬇  downloading the workshop data bundle …")
                _fetch(u, bundle)
            with zipfile.ZipFile(bundle) as z:      # flatten any folder inside the zip
                for member in z.namelist():
                    if os.path.basename(member) == name:
                        with z.open(member) as src, open(name, "wb") as dst:
                            dst.write(src.read())
                        return name
            print(f"  ({name} was not inside the bundle)")
            return None
        print(f"⬇  downloading {name} …")
        _fetch(u.rstrip("/") + "/" + name, name)
        return name
    except Exception as e:
        print(f"  (download failed: {e})")
        for leftover in (name, "_workshop_data.zip"):
            if os.path.exists(leftover) and os.path.getsize(leftover) == 0:
                os.remove(leftover)
        return None

def _cache_to_drive(name, data):
    cache = _drive_cache()
    if cache:
        dest = os.path.join(cache, name)
        data.to_csv(dest, index=False)
        print(f"  💾 saved to Google Drive ({dest}) — no need to fetch it again.")

def load_csv(name):
    """Load a data CSV: looks on disk, then downloads it from WORKSHOP_DATA_URL, then checks
    your Google-Drive cache, and only as a last resort asks you to upload it — in which case
    it saves a copy to Drive so you never have to upload it twice."""
    p = _find(name)                       # 1. already on disk?
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    p = _try_download(name)               # 2. the built-in download link
    if p:
        print(f"✓ loaded {name}")
        return pd.read_csv(p)
    p = _find_in_drive(name)              # 3. a copy you saved on a previous run
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    try:                                  # 4. last resort: upload it by hand
        from google.colab import files
        print(f"⤴  Upload {name} just once — I'll save it so the other notebooks open it automatically:")
        up = files.upload()
        fname = list(up.keys())[0]
        data = pd.read_csv(fname)
        _cache_to_drive(name, data)
        return data
    except Exception:
        raise FileNotFoundError(
            f"Could not find {name}. Either paste your download link into WORKSHOP_DATA_URL at "
            f"the top of this cell, or put the CSV next to this notebook / in a data/ folder."
        )


## 🧱 When does pandas hit a wall?

A pandas `DataFrame` lives **entirely in RAM**, and operations often make temporary copies — a good rule
of thumb is you need **≈ 5–10 × the raw file size** in free memory. So a 4 GB CSV can exhaust a 16–32 GB
laptop. Warning signs: `MemoryError`, the machine **swapping** to disk, or everything crawling.

In [2]:
# How much memory does our (small) table really use? deep=True counts the strings too.
df = load_csv("sepsis_timeseries.csv")
mem_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"{len(df):,} rows × {df.shape[1]} cols  →  {mem_mb:.1f} MB in RAM")
print(f"Rule of thumb: comfortably handle a file if you have ~5–10× its size in free RAM.")
print(f"So on a 16 GB laptop, plain pandas is happy up to roughly ~1–3 GB of CSV.")

✓ loaded data/sepsis_timeseries.csv
37,704 rows × 53 cols  →  15.7 MB in RAM
Rule of thumb: comfortably handle a file if you have ~5–10× its size in free RAM.
So on a 16 GB laptop, plain pandas is happy up to roughly ~1–3 GB of CSV.


## 🔧 First — squeeze more out of pandas (often enough!)

Before switching tools, three cheap wins:

**1. Load less / use smaller dtypes.** Read only the columns you need (`usecols`), and **downcast**:
`float64→float32`, `int64→int32`, and repeated strings → `category`. Often halves memory.

In [3]:
small = df.copy()
for c in small.select_dtypes("float64"): small[c] = pd.to_numeric(small[c], downcast="float")
for c in small.select_dtypes("int64"):   small[c] = pd.to_numeric(small[c], downcast="integer")
small["gender"] = small["gender"].astype("category")
after = small.memory_usage(deep=True).sum() / 1e6
print(f"memory: {mem_mb:.1f} MB  →  {after:.1f} MB  ({100*(1-after/mem_mb):.0f}% smaller) just from dtypes")

memory: 15.7 MB  →  7.2 MB  (54% smaller) just from dtypes


**2. Use a columnar format — Parquet, not CSV.** Parquet is compressed, typed, and reads only the
columns you ask for. It's usually **much smaller and far faster** than CSV (and preserves dtypes).

In [4]:
import os
_ensure({"pyarrow": "pyarrow"})   # Parquet needs an engine: Colab ships it, a local venv may not

df.to_parquet("_demo.parquet")
df.to_csv("_demo.csv", index=False)
csv_mb = os.path.getsize("_demo.csv") / 1e6
pq_mb  = os.path.getsize("_demo.parquet") / 1e6
print(f"CSV: {csv_mb:.1f} MB   Parquet: {pq_mb:.1f} MB   →  {csv_mb/pq_mb:.1f}× smaller")
# read back just two columns — Parquet doesn't touch the rest
two = pd.read_parquet("_demo.parquet", columns=["icustayid", "Arterial_lactate"])
print("read only 2 columns from Parquet:", two.shape)
os.remove("_demo.parquet"); os.remove("_demo.csv")

CSV: 9.5 MB   Parquet: 2.5 MB   →  3.8× smaller
read only 2 columns from Parquet: (37704, 2)


**3. Stream it in chunks.** If a CSV won't fit, read it in pieces with `chunksize` and aggregate as
you go — constant memory, whatever the file size.

In [5]:
df.to_csv("_big.csv", index=False)
# Compute a per-patient max lactate WITHOUT ever holding the whole file in memory:
running = None
for chunk in pd.read_csv("_big.csv", usecols=["icustayid", "Arterial_lactate"], chunksize=5000):
    part = chunk.groupby("icustayid")["Arterial_lactate"].max()
    running = part if running is None else pd.concat([running, part]).groupby(level=0).max()
print(f"processed the file in chunks → {len(running):,} patients, no full load")
os.remove("_big.csv")

processed the file in chunks → 1,696 patients, no full load


## 🚀 When pandas truly isn't enough — the alternatives

| Tool | What it is | Reach for it when |
|---|---|---|
| **[Polars](https://docs.pola.rs/)** | pandas-like DataFrame, multi-threaded, **lazy** + streaming (Rust) | You want a big speed/memory win with familiar code; single machine, up to *bigger-than-RAM* via streaming |
| **[DuckDB](https://duckdb.org/docs/current/)** | in-process **SQL** engine over CSV/Parquet, out-of-core | You think in SQL and want to query files **larger than RAM** on one machine (integrates with pandas/Arrow) |
| **[Dask](https://docs.dask.org/en/stable/dataframe.html)** | parallel, partitioned **pandas API** | You want the pandas API to scale across cores or a cluster / bigger-than-RAM |
| **[Apache Spark (PySpark)](https://spark.apache.org/docs/latest/api/python/index.html)** | distributed cluster compute | **Terabytes+**, many machines, production data engineering |
| **[Vaex](https://vaex.io/docs/index.html)** | out-of-core DataFrames for exploration/plots | Interactively explore/visualise **billions** of rows |
| **A database / warehouse** | Postgres, BigQuery, Snowflake… | Push the heavy joins/aggregations to where the data already lives |

A few one-liners (install first, e.g. `pip install polars duckdb`):
```python
# Polars — lazy scan, filter, aggregate, only materialise the result:
import polars as pl
pl.scan_parquet("big.parquet").filter(pl.col("SOFA") > 6).group_by("icustayid").agg(pl.col("Arterial_lactate").max()).collect()

# DuckDB — SQL directly on files bigger than RAM (no full load):
import duckdb
duckdb.sql("SELECT icustayid, MAX(Arterial_lactate) FROM 'big.parquet' GROUP BY icustayid").df()

# Dask — pandas API, partitioned across cores/cluster:
import dask.dataframe as dd
dd.read_parquet("big/*.parquet").groupby("icustayid").Arterial_lactate.max().compute()
```

👇 One-liners are easy. The rest of this notebook does the **hard** thing instead: the *whole*
cleaning pipeline from Notebook 02, written once per engine, on the same data, timed.

---

## 🧪 The same cleaning, in four engines — and what each one costs

Everything above is theory. Now the practical version: we take the **`clean()` pipeline you already
wrote in [Notebook 02](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/02_data_cleaning.ipynb)**
and rewrite it in each engine. Same input, same six steps, same output — only the syntax and the
clock change.

| | Step | Why it's interesting on a big-data engine |
|---|---|---|
| 1 | `Temp_C` if plausible, else convert `Temp_F` | trivially parallel — row-by-row |
| 2 | impossible vitals → missing, then **per-patient ffill → bfill** | needs **ordering inside each patient**: a window function, and the hard part for a distributed engine |
| 3 | winsorise 4 labs to the 1st/99th percentile | needs a **global statistic** — one pass over *all* the data before you can clip anything |
| 4 | drop exact duplicate rows | needs to compare rows that may sit in different files |
| 5 | `re_admission` → int | trivial |
| 6 | drop the near-empty `SaO2` column | trivial |

📚 **The documentation to keep open** (each engine's own page for exactly what we use here):

| Engine | Start here | The page that matters for this pipeline |
|---|---|---|
| pandas | [pandas.pydata.org/docs](https://pandas.pydata.org/docs/) | [Scaling to large datasets](https://pandas.pydata.org/docs/user_guide/scale.html) |
| Polars | [docs.pola.rs](https://docs.pola.rs/) | [Lazy API user guide](https://docs.pola.rs/user-guide/lazy/) · [Python API reference](https://docs.pola.rs/api/python/stable/reference/index.html) |
| DuckDB | [duckdb.org/docs](https://duckdb.org/docs/current/) | [Window functions — `IGNORE NULLS`](https://duckdb.org/docs/current/sql/functions/window_functions) · [Python client](https://duckdb.org/docs/current/clients/python/overview) |
| Dask | [docs.dask.org](https://docs.dask.org/en/stable/dataframe.html) | [DataFrame best practices](https://docs.dask.org/en/stable/dataframe-best-practices.html) |
| Spark | [PySpark docs](https://spark.apache.org/docs/latest/api/python/index.html) | window functions + `approxQuantile` |
| Vaex | [vaex.io/docs](https://vaex.io/docs/index.html) | (not used below — it is built for exploring/plotting, not per-group filling) |

> ⏱️ **About the timings.** Our real file is only ~38k rows — far too small to tell engines apart — so
> the next cell replicates it into **20 Parquet files (~750k rows)**. Every number you get is from
> *your* machine (a Colab VM has 2 slow cores; a laptop has more): read the **ratios**, not the
> seconds, and expect ±30% between runs.

In [6]:
# === build a bigger, realistic benchmark dataset ==============================
_ensure({"polars": "polars", "duckdb": "duckdb", "dask": "dask[dataframe]"})
import time, glob, shutil

SCALE = 20                 # 20 copies ≈ 750k rows. Try 5 on a small machine, 60 for a real workout.
BENCH = "_bench_parts"     # a *folder* of Parquet files — that is how big data actually arrives

raw = load_csv("sepsis_timeseries.csv")
if os.path.isdir(BENCH):
    shutil.rmtree(BENCH)
os.makedirs(BENCH)
for k in range(SCALE):
    part = raw.copy()
    part["icustayid"] = part["icustayid"] + k * 1_000_000     # each file gets its own patients
    part.to_parquet(f"{BENCH}/part_{k:02d}.parquet", index=False)

GLOB = f"{BENCH}/*.parquet"
size_mb = sum(os.path.getsize(f) for f in glob.glob(GLOB)) / 1e6
print(f"{SCALE} Parquet files · {SCALE * len(raw):,} rows · {size_mb:.1f} MB on disk")
print("Each file holds WHOLE patients — that's what lets an engine clean one file independently.")

# --- the cleaning rules, straight out of Notebook 02 --------------------------
RANGES = {"HR": (20, 300), "RR": (3, 80), "SpO2": (30, 100),
          "MeanBP": (20, 220), "SysBP": (40, 300), "Temp_C_clean": (25, 45)}
FILL   = list(RANGES) + ["Weight_kg"]                                    # per-patient ffill → bfill
WINSOR = ["Creatinine", "WBC_count", "Platelets_count", "Shock_Index"]   # clip to 1st/99th pct

# --- a tiny stopwatch: every engine below is run through this -----------------
TIMES, OUT = {}, {}

def run(engine, fn):
    """Time fn(), keep the result in OUT[engine], print a one-line summary."""
    t0 = time.perf_counter()
    OUT[engine] = fn()                      # the timing includes materialising the result
    TIMES[engine] = time.perf_counter() - t0
    n, k = OUT[engine].shape
    print(f"⏱️  {engine}: {TIMES[engine]:.2f} s  →  {n:,} rows × {k} columns")

✓ loaded data/sepsis_timeseries.csv


20 Parquet files · 754,080 rows · 50.3 MB on disk
Each file holds WHOLE patients — that's what lets an engine clean one file independently.


### 🐼 pandas — the baseline

Notebook 02's `clean()`, with one change: the percentile bounds can be **passed in** instead of
computed from the data at hand. That single argument is what makes the same function reusable by
Dask and Spark later — they must compute global statistics *before* they can clean a partition.

In [7]:
def clean_frame(d, q=None):
    """Notebook 02's clean(), as a plain pandas function.
    q = {column: (low, high)} percentile bounds; None → compute them from d itself."""
    d = d.sort_values(["icustayid", "bloc"])

    # 1. temperature: prefer plausible Celsius, else convert plausible Fahrenheit
    tc = d["Temp_C"].where(d["Temp_C"].between(25, 45))
    tf = d["Temp_F"].where(d["Temp_F"].between(90, 110))
    d["Temp_C_clean"] = tc.fillna((tf - 32) * 5 / 9)

    # 2. impossible vitals → NaN, then repair from each patient's own timeline
    for c, (lo, hi) in RANGES.items():
        d[c] = d[c].where((d[c] >= lo) & (d[c] <= hi))
    for c in FILL:
        d[c] = d.groupby("icustayid")[c].ffill()
        d[c] = d.groupby("icustayid")[c].bfill()

    # 3. winsorise skewed labs (clip, never delete rows)
    for c in WINSOR:
        lo, hi = q[c] if q else (d[c].quantile(0.01), d[c].quantile(0.99))
        d[c] = d[c].clip(lower=lo, upper=hi)

    # 4-6. duplicates, dtypes, drop the near-empty column
    d = d.drop_duplicates()
    d["re_admission"] = d["re_admission"].astype(int)
    return d.drop(columns=["SaO2"])


run("pandas", lambda: clean_frame(pd.read_parquet(BENCH)).reset_index(drop=True))

⏱️  pandas: 1.40 s  →  753,960 rows × 53 columns


### ⚡ Polars — the same steps, lazily

`scan_parquet` doesn't read anything: it builds a **query plan**. Polars then optimises the whole
pipeline and runs it multi-threaded when you finally call `.collect()`. Two things to notice:

- `.forward_fill().backward_fill().over("icustayid")` — the `.over()` is the whole per-patient
  groupby, expressed as part of the expression rather than as a separate step.
- the percentiles force a **second pass** (`.collect()` twice): you cannot clip to a percentile you
  haven't computed yet. Every engine here pays that cost.
- ⚠️ `.quantile()` defaults to `interpolation="nearest"` in Polars but `"linear"` in pandas — pass it
  explicitly or your clipped values will quietly disagree with Notebook 02.

*(See [Lazy API](https://docs.pola.rs/user-guide/lazy/) · [expressions reference](https://docs.pola.rs/api/python/stable/reference/expressions/index.html).)*

In [8]:
import polars as pl

def clean_polars():
    lf = pl.scan_parquet(GLOB).sort(["icustayid", "bloc"])          # nothing read yet — just a plan

    # 1. temperature
    tc = pl.when(pl.col("Temp_C").is_between(25, 45)).then(pl.col("Temp_C"))
    tf = pl.when(pl.col("Temp_F").is_between(90, 110)).then((pl.col("Temp_F") - 32) * 5 / 9)
    lf = lf.with_columns(tc.otherwise(tf).alias("Temp_C_clean"))

    # 2. impossible vitals → null, then per-patient forward/backward fill
    lf = lf.with_columns([pl.when(pl.col(c).is_between(lo, hi)).then(pl.col(c)).alias(c)
                          for c, (lo, hi) in RANGES.items()])
    lf = lf.with_columns([pl.col(c).forward_fill().backward_fill().over("icustayid")
                          for c in FILL])

    # 3. winsorise — a global statistic, so this is a separate pass over the data
    q = lf.select([pl.col(c).quantile(p, interpolation="linear").alias(f"{c}{p}")
                   for c in WINSOR for p in (0.01, 0.99)]).collect().row(0, named=True)
    lf = lf.with_columns([pl.col(c).clip(q[f"{c}0.01"], q[f"{c}0.99"]) for c in WINSOR])

    # 4-6. duplicates, dtypes, drop the near-empty column
    lf = lf.unique().with_columns(pl.col("re_admission").cast(pl.Int32)).drop("SaO2")
    return lf.collect()                                             # ← now it actually runs

run("Polars", clean_polars)

⏱️  Polars: 0.17 s  →  753,960 rows × 53 columns


### 🦆 DuckDB — the same steps, in SQL

No DataFrame at all: one query, straight over the Parquet files, never loading them fully.

- **forward-fill in SQL** is `last_value(x IGNORE NULLS) OVER (PARTITION BY patient ORDER BY bloc
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`; back-fill is the same with `first_value` and
  the window running *forward*. `COALESCE(ffill, bfill)` reproduces Notebook 02 exactly.
- `SELECT * REPLACE (...)` and `* EXCLUDE (...)` are DuckDB conveniences that let you rewrite 6 of 53
  columns without typing the other 47.
- We build the SQL with an f-string from the same `RANGES`/`FILL`/`WINSOR` dicts — add
  `print(sql)` inside the function to read the generated query.

*(See [window functions](https://duckdb.org/docs/current/sql/functions/window_functions) · [Python client](https://duckdb.org/docs/current/clients/python/overview).)*

In [9]:
import duckdb

def clean_duckdb():
    # build the column lists once — same dicts as every other engine
    rng  = ",\n        ".join(f"CASE WHEN {c} BETWEEN {lo} AND {hi} THEN {c} END AS {c}"
                              for c, (lo, hi) in RANGES.items())
    fill = ",\n        ".join(f"COALESCE(last_value({c} IGNORE NULLS) OVER past,"
                              f" first_value({c} IGNORE NULLS) OVER future) AS {c}" for c in FILL)
    wins = ",\n        ".join(f"greatest(least({c}, q.{c}_hi), q.{c}_lo) AS {c}" for c in WINSOR)
    quant = ",\n           ".join(f"quantile_cont({c}, 0.01) AS {c}_lo,"
                                  f" quantile_cont({c}, 0.99) AS {c}_hi" for c in WINSOR)

    sql = f"""
    WITH src AS (SELECT * FROM read_parquet('{GLOB}')),
      q AS (SELECT {quant} FROM src),                          -- 3. the global percentiles
      tempc AS (                                               -- 1. temperature
        SELECT *, COALESCE(CASE WHEN Temp_C BETWEEN 25 AND 45 THEN Temp_C END,
                           (CASE WHEN Temp_F BETWEEN 90 AND 110 THEN Temp_F END - 32) * 5.0/9.0)
                  AS Temp_C_clean FROM src),
      ranged AS (SELECT * REPLACE ({rng}) FROM tempc),         -- 2a. impossible → NULL
      filled AS (                                              -- 2b. per-patient ffill → bfill
        SELECT * REPLACE ({fill}) FROM ranged
        WINDOW past   AS (PARTITION BY icustayid ORDER BY bloc
                          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
               future AS (PARTITION BY icustayid ORDER BY bloc
                          ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING))
    SELECT DISTINCT f.* EXCLUDE (SaO2)                         -- 4. + 6.
                    REPLACE ({wins},                           -- 3. clip
                             CAST(f.re_admission AS INTEGER) AS re_admission)   -- 5.
    FROM filled f, q
    """
    return duckdb.sql(sql).df()      # .df() → pandas; .arrow() / .pl() also work, zero-copy

run("DuckDB", clean_duckdb)

⏱️  DuckDB: 1.01 s  →  753,960 rows × 53 columns


### 🧩 Dask — your pandas code, once per partition

Dask doesn't rewrite anything: it **reuses `clean_frame` unchanged** and applies it to each Parquet
file in parallel. That is the whole selling point — and the whole trap:

- ✅ it works here **only because each file holds whole patients**. If one patient's rows were split
  across two files, the per-patient forward-fill would silently be wrong. In real life you first
  `set_index("icustayid")` (an expensive shuffle) — see
  [best practices](https://docs.dask.org/en/stable/dataframe-best-practices.html).
- the percentiles must be computed **globally, before** the partitions are cleaned — one
  `.compute()` for the statistics, one for the cleaning. Dask's `.quantile()` is approximate by
  default; on this data it lands on the same values as pandas.

In [10]:
import dask.dataframe as dd

def clean_dask():
    ddf = dd.read_parquet(BENCH)                      # 1 partition per Parquet file
    qs = ddf[WINSOR].quantile([0.01, 0.99]).compute()          # pass 1: global statistics
    q = {c: (qs.loc[0.01, c], qs.loc[0.99, c]) for c in WINSOR}
    return (ddf.map_partitions(clean_frame, q)                 # pass 2: your pandas code, in parallel
               .compute().reset_index(drop=True))

print("partitions:", dd.read_parquet(BENCH).npartitions)
run("Dask", clean_dask)

partitions: 20


⏱️  Dask: 0.89 s  →  753,960 rows × 53 columns


### ✨ Spark — the same pipeline, cluster-shaped *(optional)*

Off by default: PySpark is a ~300 MB install and starts a JVM, so it costs ~2 minutes for a dataset
this size — and it will be the **slowest** engine here, because on 750k rows you pay for the cluster
machinery without ever using it. Flip `RUN_SPARK = True` if you want to watch it work.

The translation is nearly one-to-one with DuckDB — `Window.partitionBy(...).orderBy(...)` plus
`F.last(..., ignorenulls=True)` *is* forward-fill. The one genuine difference: `approxQuantile` is
**approximate** (that's the price of doing it across a cluster), so Spark's clipped maxima can differ
from pandas in the last decimal.

In [11]:
RUN_SPARK = False          # ← set to True to actually run it (installs PySpark, needs Java, ~2 min)

def clean_spark():
    from pyspark.sql import functions as F, Window
    s = spark.read.parquet(BENCH)

    # 1. temperature
    tc = F.when(F.col("Temp_C").between(25, 45), F.col("Temp_C"))
    tf = F.when(F.col("Temp_F").between(90, 110), (F.col("Temp_F") - 32) * 5 / 9)
    s = s.withColumn("Temp_C_clean", F.coalesce(tc, tf))

    # 2. impossible vitals → null, then per-patient ffill → bfill via window functions
    for c, (lo, hi) in RANGES.items():
        s = s.withColumn(c, F.when(F.col(c).between(lo, hi), F.col(c)))
    past   = Window.partitionBy("icustayid").orderBy("bloc").rowsBetween(Window.unboundedPreceding, 0)
    future = Window.partitionBy("icustayid").orderBy("bloc").rowsBetween(0, Window.unboundedFollowing)
    for c in FILL:
        s = s.withColumn(c, F.coalesce(F.last(c, ignorenulls=True).over(past),
                                       F.first(c, ignorenulls=True).over(future)))

    # 3. winsorise — approxQuantile, not an exact percentile
    for c, (lo, hi) in zip(WINSOR, s.approxQuantile(WINSOR, [0.01, 0.99], 0.0001)):
        s = s.withColumn(c, F.least(F.greatest(F.col(c), F.lit(lo)), F.lit(hi)))

    # 4-6.
    return (s.dropDuplicates()
             .withColumn("re_admission", F.col("re_admission").cast("int"))
             .drop("SaO2")
             .toPandas())

if RUN_SPARK:
    _ensure({"pyspark": "pyspark"})
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.master("local[*]").appName("clean").getOrCreate()
    run("Spark", clean_spark)
else:
    print("Spark skipped (RUN_SPARK = False) — read the code above instead:")
    print("  Window(...).rowsBetween(...) + F.last(ignorenulls=True)  ==  per-patient forward-fill.")

Spark skipped (RUN_SPARK = False) — read the code above instead:
  Window(...).rowsBetween(...) + F.last(ignorenulls=True)  ==  per-patient forward-fill.


### 🏁 The scoreboard — and did they all agree?

Speed is worthless if the answers differ. Alongside the seconds we fingerprint each result: row
count, column count, two means, two maxima, and the total number of missing cells. **All of them must
match**, or one of the translations is wrong.

In [12]:
def as_pandas(x):
    """Polars DataFrame / Arrow table → pandas, so we can compare like with like."""
    return x.to_pandas() if type(x).__module__.startswith(("polars", "pyarrow")) else x

def fingerprint(x):
    d = as_pandas(x)
    return pd.Series({"rows": len(d), "cols": d.shape[1],
                      "HR mean": round(float(d["HR"].mean()), 3),
                      "SysBP mean": round(float(d["SysBP"].mean()), 3),
                      "Temp_C_clean max": round(float(d["Temp_C_clean"].max()), 2),
                      "Creatinine max": round(float(d["Creatinine"].max()), 2),
                      "missing cells": int(d.isna().sum().sum())})

board = pd.DataFrame({e: fingerprint(o) for e, o in OUT.items()}).T
board.insert(0, "seconds", pd.Series(TIMES).round(2))
board.insert(1, "vs pandas", (TIMES["pandas"] / pd.Series(TIMES)).round(1).astype(str) + "×")

agree = board.drop(columns=["seconds", "vs pandas"]).nunique().eq(1).all()
print("✅ every engine produced an identical table" if agree else
      "⚠️ the engines DISAGREE — one translation is wrong (look at the columns below)")
display(board)

shutil.rmtree(BENCH)      # tidy up the benchmark files (comment out to keep experimenting)

✅ every engine produced an identical table


,seconds,vs pandas,rows,cols,HR mean,SysBP mean,Temp_C_clean max,Creatinine max,missing cells
pandas,1.40,1.0×,753960.0,53.0,84.519,120.967,40.0,12.44,366740.0
Polars,0.17,8.3×,753960.0,53.0,84.519,120.967,40.0,12.44,366740.0
DuckDB,1.01,1.4×,753960.0,53.0,84.519,120.967,40.0,12.44,366740.0
Dask,0.89,1.6×,753960.0,53.0,84.519,120.967,40.0,12.44,366740.0


### 📖 How to read that table

- **Polars is the free lunch** at this size: same laptop, same result, several times faster, and the
  code is the closest to what you already write. If you change one thing after this notebook, make it
  this one.
- **DuckDB is competitive and never loaded the files** — it streamed them. That is why it keeps
  working when the data is bigger than RAM, and pandas doesn't.
- **Dask is a coin-flip against pandas here** — it runs the *identical* function, but at 750k rows
  the scheduling overhead roughly cancels the parallel speed-up, so it lands either side of 1× from
  run to run. Dask wins when the data no longer fits in memory, not when it is merely large — and
  **Spark** starts winning one or two orders of magnitude further out again.
- **Everything is a two-pass pipeline** because of one line: winsorising to a *global* percentile.
  Notice how the engines differ in what they're willing to do about it (exact vs approximate).
- The obvious follow-up: raise `SCALE` to 60 and re-run. The ratios move — and *that* is the real
  lesson. "Which tool is fastest" only ever has an answer for a given size on a given machine.

> ⚠️ **Careful with per-patient operations at scale.** Forward-filling inside a patient assumes all of
> that patient's rows are (a) in the same partition and (b) sorted by time. pandas gives you both for
> free; every distributed engine makes you ask. Getting this wrong doesn't crash — it just quietly
> produces a slightly wrong dataset, which is exactly the failure mode Notebook 08 warns about.

## 🧭 A rough decision guide

| Data size (single machine, ~16 GB RAM) | Use |
|---|---|
| Fits in RAM (≲ 1–3 GB CSV) | **pandas** (or **Polars** for speed) |
| Bigger than RAM, one machine (~GBs–100s GB) | **DuckDB** or **Polars (streaming)**; chunked pandas; convert to **Parquet** |
| Won't fit + needs many cores / a cluster | **Dask** |
| Terabytes, many machines, production | **Spark**, or a **cloud warehouse** (BigQuery/Snowflake) |

> 🧠 **Takeaway.** 90% of "big data" problems on a laptop are solved by **Parquet + dtype downcasting +
> chunking**, or by switching to **Polars/DuckDB** — no cluster needed. Reach for Spark only when you
> genuinely have many machines' worth of data.

## 🗂️ Where do the really big datasets come from? *(sources)*

**Large public ICU / EHR / health datasets** (most need registration + a data-use agreement):
- **eICU-CRD** — 200k+ ICU stays, multi-centre US · **HiRID**, **AmsterdamUMCdb** — high-resolution ICU
- Large open **critical-care databases** and **waveform archives** — ask your library or research office;
  most require a short registration and a data-use agreement
- **UK Biobank** — 500k participants, genetics + imaging + EHR · **All of Us** (NIH) — 1M+ participants
- **SEER** — cancer registry · **OMOP CDM / OHDSI** — federated EHR across institutions
- Your own hospital's **data warehouse** (often the biggest — and messiest — of all)

These arrive as **CSV, Parquet, HDF5, Arrow, or in a database** — for anything above a few GB, prefer
Parquet/database access over one giant CSV.

## 📚 Further reading
- pandas — [Scaling to large datasets](https://pandas.pydata.org/docs/user_guide/scale.html) (official guide)
- [Polars](https://docs.pola.rs/) — [lazy API](https://docs.pola.rs/user-guide/lazy/) · [Python API reference](https://docs.pola.rs/api/python/stable/reference/index.html)
- [DuckDB](https://duckdb.org/docs/current/) — [Python client](https://duckdb.org/docs/current/clients/python/overview) · [window functions](https://duckdb.org/docs/current/sql/functions/window_functions)
- [Dask DataFrame](https://docs.dask.org/en/stable/dataframe.html) — [best practices](https://docs.dask.org/en/stable/dataframe-best-practices.html) · [PySpark](https://spark.apache.org/docs/latest/api/python/index.html) · [Vaex](https://vaex.io/docs/index.html)
- The columnar formats underneath all of them: [Apache Parquet](https://parquet.apache.org/docs/) · [Apache Arrow](https://arrow.apache.org/docs/)

## ✅ Recap
- pandas is a **RAM** tool: great to ~1–3 GB on a laptop, then it struggles.
- Stretch it with **`usecols` + dtype downcasting**, **Parquet**, and **`chunksize`** streaming.
- Beyond that: **Polars / DuckDB** (one machine, out-of-core), **Dask** (parallel pandas), **Spark** (cluster).
- The *same* cleaning pipeline runs in all of them and gives the **same table** — what changes is the
  syntax, the runtime, and how careful you must be about per-patient ordering and global statistics.
- Store big data as **Parquet**, not CSV — and let a **database/warehouse** do the heavy lifting when you can.

<!-- nav-footer -->
---

### ➡️ Next up — Notebook 13: Fairness & subgroup performance

<a href="https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/13_fairness_and_subgroups.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook 13 in Colab" height="32"/></a>

👉 **[Continue to Notebook 13 — Fairness & subgroup performance](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/13_fairness_and_subgroups.ipynb)**

[⬅ Back to Notebook 11](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/11_reinforcement_learning_qlearning.ipynb) · [🗺️ Course index](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb) · [📁 The course on GitHub](https://github.com/lorenzkap/ML2026)